# ROGII — An Honest CV Harness & the Formation-Surface Geometry

Most public notebooks here are *blends of blends*. This one is different: it is a **methodology** notebook. It answers three questions that decide whether you medal on the **private** leaderboard (Aug 5), not the public one:

1. **What is the real baseline?** (Spoiler: `last-known-TVT` = 15.91 RMSE, and it is shockingly hard to beat.)
2. **Where does the signal actually live?** (In the *formation contacts* — an almost-exact geometric identity, not in clever GR alignment.)
3. **Why is the public LB a 3-well mirage**, and how do you build a CV you can trust instead?

If you take one thing away: **trust a well-grouped CV, distrust the public LB.** There are only 3 visible test wells; the private set is unseen wells. A submission tuned on the public 3 will not transfer.

In [ ]:
import numpy as np, pandas as pd, glob, os
from pathlib import Path

# auto-discover the competition data mount (robust to Kaggle's path layout)
ROOT = Path(os.path.dirname(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)[0]))
print('data root:', ROOT)
train_wells = sorted(p.stem.replace('__horizontal_well','')
                     for p in (ROOT/'train').glob('*__horizontal_well.csv'))
test_wells  = sorted(p.stem.replace('__horizontal_well','')
                     for p in (ROOT/'test').glob('*__horizontal_well.csv'))
print(f'train wells: {len(train_wells)}   test wells: {len(test_wells)}')
print('test well ids:', test_wells)

## 1. The data shape: a solved prefix and a hidden tail

Each horizontal well is a sequence along measured depth (`MD`). The interpreter has already picked `TVT` up to some point — that is `TVT_input`. After the *prediction start* it becomes `NaN`, and those tail rows are what we predict. The columns available **at test time** are only `MD, X, Y, Z, GR, TVT_input` — the formation columns (`ANCC … BUDA`) are train-only, which matters later.

In [ ]:
def load(wid, split):
    hw = pd.read_csv(ROOT/split/f'{wid}__horizontal_well.csv')
    tw = pd.read_csv(ROOT/split/f'{wid}__typewell.csv')
    return hw, tw

hw, tw = load(train_wells[0], 'train')
print('horizontal columns:', list(hw.columns))
print('typewell columns  :', list(tw.columns))
n_pre = hw['TVT_input'].notna().sum()
print(f'\nexample well {train_wells[0]}: {len(hw)} rows, '
      f'{n_pre} solved prefix, {len(hw)-n_pre} hidden tail')

## 2. The baseline everyone underestimates

The single most important number in this competition: predict the **last known TVT**, flat, for the whole tail. Geosteered wells are *steered to stay in zone*, so TVT mean-reverts — a flat line is a strong prior. Let's measure it across all 773 wells (this is the honest pooled RMSE, the same metric the LB uses).

In [ ]:
def pooled_rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred)-np.asarray(true))**2)))

errs = []
for wid in train_wells:
    hw,_ = load(wid,'train')
    pre = hw[hw['TVT_input'].notna()]; tail = hw[hw['TVT_input'].isna()]
    if len(tail)==0 or len(pre)<10 or hw['TVT'].isna().all():
        continue
    last = pre['TVT_input'].iloc[-1]
    errs.append((tail['TVT'].values - last))
errs = np.concatenate(errs)
print(f'FLAT last-known-TVT pooled RMSE: {np.sqrt((errs**2).mean()):.3f} ft')

**15.91 ft.** Every leaderboard point below ~9 is hard-won correction on top of this flat line. Naive geometric extrapolation (continue the prefix slope) *increases* error — slopes explode beyond a few hundred feet. So the gain has to come from elsewhere.

## 3. Where the signal lives: a formation-surface identity

The hidden lever in this dataset is geometric. For each well, `TVT` and a formation contact (say `ANCC`) relate through the trajectory `Z` by an almost-exact identity:

$$ \text{TVT} \;=\; (\text{ANCC} - Z) \;+\; c_{\text{well}} $$

where `c_well` is a per-well constant you can calibrate on the solved prefix. Let's check how exact it is.

In [ ]:
stds = []
for wid in train_wells[:200]:
    hw,_ = load(wid,'train')
    pre = hw[hw['TVT_input'].notna()]
    if 'ANCC' not in hw or pre['ANCC'].isna().all():
        continue
    resid = pre['TVT_input'] - (pre['ANCC'] - pre['Z'])
    stds.append(resid.std())
print(f'residual std of TVT-(ANCC-Z) on the prefix: '
      f'median {np.median(stds):.4f} ft over {len(stds)} wells')

**~0.01 ft.** The identity is essentially exact within a well. So the task *reduces* to: estimate the formation surface at the hidden tail. The catch — the formation columns are train-only, so for a hidden well you must **reconstruct the surface from neighbouring wells** (spatial interpolation in X/Y), then snap back to TVT with the prefix-calibrated `c`. This is why the strong public pipelines are built around offset-well formation surfaces, particle filters over the contact, and beam search — they are all estimating that surface under uncertainty.

> Practical note: interpolating the surface *directly* and predicting from it is **worse than flat** (the cross-well surface is spiky; even the nearest offset well at <100 ft loses to the flat prior). What works is feeding the surface estimate **and its reliability** (neighbour distance, vote spread) into a gradient-boosted residual model that learns *when to trust it*.

## 4. The CV that actually transfers: GroupKFold by well

Rows within a well are massively autocorrelated. A random KFold leaks future rows of the same well into training and gives a fantasy score. **Split by well.** Here is a minimal, honest harness you can drop into any model: it builds last-known-TVT residual features, validates with `GroupKFold(well)`, and reports the pooled RMSE the LB uses.

In [ ]:
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

rows = []
for wid in train_wells:
    hw,_ = load(wid,'train')
    pre = hw[hw['TVT_input'].notna()]; tail = hw[hw['TVT_input'].isna()]
    if len(tail)==0 or len(pre)<10 or hw['TVT'].isna().all():
        continue
    last_md = pre['MD'].iloc[-1]; last_tvt = pre['TVT_input'].iloc[-1]; last_z = pre['Z'].iloc[-1]
    k = min(200, len(pre))
    slope = np.polyfit(pre['MD'].iloc[-k:], pre['TVT_input'].iloc[-k:], 1)[0]
    t = tail.copy()
    t['well']=wid; t['md_from_cut']=t['MD']-last_md; t['z_from_last']=t['Z']-last_z
    t['slope200']=slope; t['last_tvt']=last_tvt
    t['resid']=t['TVT']-last_tvt
    rows.append(t[['well','md_from_cut','z_from_last','GR','slope200','X','Y','Z','last_tvt','resid','TVT']])
df = pd.concat(rows, ignore_index=True)
df = df.iloc[::5]  # subsample for a fast demo
print('training rows (subsampled):', len(df))

In [ ]:
FEATS=['md_from_cut','z_from_last','GR','slope200','X','Y','Z']
y = df['resid'].values; groups = df['well'].values
oof = np.zeros(len(df))
for tr,va in GroupKFold(5).split(df, y, groups):
    m = lgb.LGBMRegressor(objective='l2', num_leaves=127, learning_rate=0.05,
                          n_estimators=500, min_child_samples=200,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          n_jobs=-1, verbosity=-1)
    m.fit(df[FEATS].iloc[tr], y[tr])
    oof[va] = m.predict(df[FEATS].iloc[va])
pred = df['last_tvt'].values + oof
print(f'GroupKFold CV — geometry-only LGB: {pooled_rmse(pred, df["TVT"]):.3f} ft')
print(f'(flat baseline on same rows:      {pooled_rmse(df["last_tvt"], df["TVT"]):.3f} ft)')

A geometry-only GBM (no formation surface, no GR alignment) barely moves off flat — confirming the signal is in the formation surface, not the trajectory. Add the offset-well surface estimate + its reliability as features and this drops into the ~13 range; the full public pipelines (PF/beam over the contact + offset-well surfaces + stacking) reach ~9 pooled CV.

## 5. Why the public LB is a 3-well mirage

There are **3 visible test wells**. The private leaderboard scores **unseen wells**. Two consequences:

- A public score can sit well below a pipeline's true pooled CV simply because those 3 wells are favourable. (A pipeline with ~9 ft CV can show ~7.6 public.) That gap is **not** skill — it is sampling.
- Tuning weights/thresholds to the public 3 is overfitting to noise. On Aug 5 the cluster reshuffles toward each pipeline's real CV.

**The defensible strategy:** measure everything on `GroupKFold(well)`, prefer simple robust blends, and choose your two final submissions on CV — not on which public number is highest. The harness above is all you need to do that honestly.

---
*If this saved you a few wasted submissions, an upvote is appreciated — and good luck on the private board.* 🛢️